Load libraries and functions

In [1]:
from poisson_bcann_vi import *
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from poisson_bcann_hmc_nuts import (
    stratified_subsample, make_chain_inits, run_nuts, time_nuts, get_device, 
)
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from poisson_nuts_parallel import run_nuts_parallel
from bcann_neutra import (
    neutra_fit_guide, run_neutra_parallel, time_neutra
)

seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perform train/validation/test split

In [2]:
device = torch.device("cuda:1")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)

train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)

val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
)

learn_loader = DeviceLoader(
        X_learn, y_learn, torch.log(exp_learn),
        batch_size= 10000
)

Start PoissonBCANN, and train the IAF with guide function

In [3]:
network_architecture = [20, 15, 10]

# flow (step 2)
NUM_TRANSFORMS = 2        # IAF layers; 0 -> diagonal Gaussian (NeuTra only rescales the space)
FLOW_HIDDEN = None       # IAF hidden widths; None -> [latent_dim]. Try [512] if the flow is slow
SVI_BATCH = None          # None -> full-batch ELBO on the subsample
SVI_STEPS = 5000
SVI_LR, SVI_LR_FINAL = 5e-3, 5e-4

glm_mle = poisson_fit_glm_mle(X_learn, y_learn, torch.log(exp_learn), random_state=seed)

with torch.device(device):
        poisson_bcann = PoissonBCANN(
                in_features= 48,
                neural_net_params={"hidden_activation_nodes": network_architecture,
                                    "hidden_activations": nn.Tanh,
                                    "prior_scale": 1.0},
            )
        poisson_guide, poisson_history = neutra_fit_guide(
            poisson_bcann, 
            X_learn, 
            y_learn, 
            torch.log(exp_learn), 
            glm_mle=tuple(t.to(device) for t in glm_mle),
            num_transforms=NUM_TRANSFORMS, hidden_dim= FLOW_HIDDEN,
            num_steps=SVI_STEPS, lr=SVI_LR, lr_final=SVI_LR_FINAL, batch_size=10000,
            report_every=max(SVI_STEPS // 20, 1), random_state=seed
        )


step 250 | -ELBO/n: 0.2053 | 7s
step 500 | -ELBO/n: 0.2050 | 15s
step 750 | -ELBO/n: 0.2040 | 22s
step 1000 | -ELBO/n: 0.2046 | 29s
step 1250 | -ELBO/n: 0.2043 | 36s
step 1500 | -ELBO/n: 0.2050 | 43s
step 1750 | -ELBO/n: 0.2049 | 51s
step 2000 | -ELBO/n: 0.2038 | 58s
step 2250 | -ELBO/n: 0.2034 | 65s
step 2500 | -ELBO/n: 0.2045 | 72s
step 2750 | -ELBO/n: 0.2031 | 79s
step 3000 | -ELBO/n: 0.2043 | 86s
step 3250 | -ELBO/n: 0.2034 | 94s
step 3500 | -ELBO/n: 0.2044 | 102s
step 3750 | -ELBO/n: 0.2041 | 109s
step 4000 | -ELBO/n: 0.2038 | 117s
step 4250 | -ELBO/n: 0.2035 | 124s
step 4500 | -ELBO/n: 0.2044 | 131s
step 4750 | -ELBO/n: 0.2040 | 138s
step 5000 | -ELBO/n: 0.2045 | 145s


NeuTra hyperparameters

In [4]:
# NUTS (step 3)
CHAINS = 6
WARMUP, SAMPLES = 1000, 1000
TARGET_ACCEPT = 0.8       # raise to 0.9 for the horseshoe models if there are divergences
MAX_TREE_DEPTH = 8

nuts_kwargs = dict(
    num_warmup=WARMUP, 
    num_samples=SAMPLES, 
    target_accept=TARGET_ACCEPT,
    max_tree_depth=MAX_TREE_DEPTH, 
    init_scale=1.0,
    jit_compile = True
)


Test run to see how long each iteration takes

In [5]:
time_neutra(
    poisson_bcann, poisson_guide, 
    X_learn, 
    y_learn,
    torch.log(exp_learn),
    n_iter=20,
    device=device
)  # cost estimate first

0.41 s / iteration on cuda:1  (early warmup iterations are not representative of tree depth)


0.40987218618392945

Run Neutra with full config

In [6]:
poisson_neutra_samples, poisson_neutra_info = run_neutra_parallel(
    PoissonBCANN, 
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ), 
    poisson_guide, 
    X_learn, 
    y_learn,
    torch.log(exp_learn),
    num_chains=CHAINS, devices=device,
    out_dir=f"poisson_neutra_chains", seed=seed,
    **nuts_kwargs
)

elapsed 1:59:23
chain 5: Sample: 100%|██████████| 2000/2000 [1:59:01,  3.57s/it, step size=2.29e-02, acc. prob=0.955]


Write to disk for later uses

In [7]:
torch.save(
    {
        "samples": poisson_neutra_samples,
        "info": poisson_neutra_info,
        "config": dict(num_warmup=1000, num_samples=1000, target_accept=0.8,
                       max_tree_depth=8, seed=seed, n_chains=6),
    },
    "hmc_results/poisson_neutra_full_set.pt",
)